In [ ]:
import pandas
from matplotlib import pyplot
def fixed_diff(data):
    fixed = (data[data["id"] == "fixed"][["seed", "round", "accuracy", "loss"]]
        .rename(columns={"accuracy": "fixed_accuracy", "loss": "fixed_loss",}))
    data_diff = data.merge(fixed, on=["seed", "round"])
    data_diff["accuracy"] = data_diff["accuracy"] - data_diff["fixed_accuracy"]
    data_diff["loss"] = data_diff["loss"] - data_diff["fixed_loss"]
    data_diff = data_diff.drop(columns=["fixed_accuracy", "fixed_loss"])
    return data_diff
def flwr_accuracy_error(ax, file, start=0, key="accuracy"):
    data = pandas.read_csv(file + ".csv")
    data = data[data["round"] >= start]
    data = fixed_diff(data)
    summary = (data.groupby(["id", "round"], sort=False)[key].agg(mean="mean", std="std").reset_index())
    pyplot.figure(figsize=(11, 6))
    for id_val, group in summary.groupby("id", sort=False):
        group = group.sort_values("round")
        if id_val == "fixed":
            ax.plot(group["round"], group["mean"], label=str(id_val))
        else:
            ax.errorbar(group["round"], group["mean"], yerr=group["std"], linewidth=2, capsize=4, alpha=0.7, label=str(id_val))
    ax.set_xlabel("round")
    ax.set_ylabel(key + " diff")
    ax.grid(True, alpha=0.3)
    ax.legend()
def flwr_accuracy(ax, file, start=0, key="accuracy"):
    data = pandas.read_csv(file + ".csv")
    data = data[data["round"] >= start]
    summary = (data.groupby(["id", "round"], sort=False)[key].agg(mean="mean", std="std").reset_index())
    for id_val, group in summary.groupby("id", sort=False):
        group = group.sort_values("round")
        line, = ax.plot(group["round"], group["mean"], label=str(id_val))
        ax.fill_between(
        group["round"],
        group["mean"] - group["std"],
        group["mean"] + group["std"],
        color=line.get_color(),
        alpha=0.2
    )

    ax.set_xlabel("round")
    ax.set_ylabel(key)
    ax.grid(True, alpha=0.3)
    ax.legend()

In [ ]:
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(13, 4))
flwr_accuracy(ax1, "budget3", 160)
flwr_accuracy_error(ax2, "budget3", 160)
fig.savefig("budget3accuracy.pdf", bbox_inches="tight")

In [ ]:
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(13, 4))
flwr_accuracy(ax1, "budget3_sub", 160)
flwr_accuracy(ax2, "budget3", 160, "loss")
fig.savefig("budget3loss.pdf", bbox_inches="tight")